# Phase 3 — Faster R-CNN Defect Detection

Fine-tunes a COCO-pretrained Faster R-CNN (ResNet50-FPN backbone) on
NEU-DET bounding-box annotations. Unlike Phases 1-2 (whole-image
classification), this model localizes *where* each defect is in the
image, not just *what type* the image contains overall.

Uses `DetectionDataset` + `detection_collate_fn` (data), `build_fasterrcnn`
(model), and `fit_detection` / `evaluate_detection` (training loop with
a custom, dependency-free mAP metric) — all built in previous steps.

In [ ]:
import sys
sys.path.append('..')

import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as patches

%matplotlib inline

from src.data.detection_dataset import DetectionDataset, detection_collate_fn, CLASS_NAMES
from src.models.detection_model import build_fasterrcnn
from src.models.train_detection import fit_detection, evaluate_detection

DATA_ROOT = '../data/raw/NEU-DET'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
train_set = DetectionDataset(f'{DATA_ROOT}/train')
val_set = DetectionDataset(f'{DATA_ROOT}/validation')
print(f'Train samples: {len(train_set)} | Val samples: {len(val_set)}')

train_loader = DataLoader(train_set, batch_size=2, shuffle=True, collate_fn=detection_collate_fn)
val_loader = DataLoader(val_set, batch_size=2, shuffle=False, collate_fn=detection_collate_fn)

In [ ]:
# num_classes = 6 defect types + 1 background class
model = build_fasterrcnn(num_classes=len(CLASS_NAMES) + 1, backbone='mobilenet').to(device)

## Train

This will be noticeably slower per epoch than Phase 2's classifiers —
Faster R-CNN is a much larger, more complex model. Fewer epochs are
needed than the classifiers since the pretrained backbone already does
most of the work.

In [ ]:
history = fit_detection(
    model,
    train_loader,
    val_loader,
    device,
    class_names=CLASS_NAMES,
    epochs=8,
    lr=5e-3,
    checkpoint_path='../models/fasterrcnn.pt',
)

## Training curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'])
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_mAP'])
axes[1].set_title('Validation mAP@0.5')
axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

## Visualize predictions on a few validation images

In [ ]:
model.load_state_dict(torch.load('../models/fasterrcnn.pt', map_location=device))
model.eval()

n_show = 4
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))

with torch.no_grad():
    for i in range(n_show):
        img_tensor, target = val_set[i]
        pred = model([img_tensor.to(device)])[0]

        axes[i].imshow(img_tensor[0], cmap='gray')
        axes[i].axis('off')

        for box in target['boxes']:
            x1, y1, x2, y2 = box.tolist()
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='lime', facecolor='none')
            axes[i].add_patch(rect)

        keep = pred['scores'] > 0.5
        for box, label, score in zip(pred['boxes'][keep], pred['labels'][keep], pred['scores'][keep]):
            x1, y1, x2, y2 = box.cpu().tolist()
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='red', facecolor='none')
            axes[i].add_patch(rect)
            class_name = CLASS_NAMES[label.item() - 1]
            axes[i].text(x1, y1 - 3, f'{class_name} {score:.2f}', color='red', fontsize=8)

        axes[i].set_title(f'val[{i}]  (green=GT, red=pred)')

plt.tight_layout()
plt.show()

## Final mAP summary

In [ ]:
final_map, per_class_ap = evaluate_detection(model, val_loader, device, CLASS_NAMES)
print(f'Final val mAP@0.5: {final_map:.4f}\n')
for name, ap in per_class_ap.items():
    print(f'  {name:<16}: AP={ap:.4f}')